In [10]:
import numpy as np
import pandas as pd

In [12]:
df = pd.read_csv('diabetes.csv')

In [13]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


USING KERAS TUNER TO GET THE BEST OPTIMIZER AMONG 4 OPTIONS

In [14]:
df.corr()['Outcome']

,Outcome
Pregnancies,0.221898
Glucose,0.466581
BloodPressure,0.065068
SkinThickness,0.074752
Insulin,0.130548
BMI,0.292695
DiabetesPedigreeFunction,0.173844
Age,0.238356
Outcome,1.000000


In [15]:
X  = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

In [16]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [17]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [18]:
import tensorflow
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense

In [19]:
model = Sequential()
model.add(Dense(32, activation='relu', input_dim=8))
model.add(Dense(1, activation= 'sigmoid'))

model.compile(optimizer='Adam', loss='binary_crossentropy', metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [20]:
model.fit(X_train, y_train, batch_size=32, epochs=100, verbose=1, validation_data=(X_test, y_test))

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 56ms/step - accuracy: 0.5456 - loss: 9.3834 - val_accuracy: 0.4481 - val_loss: 6.4838
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5163 - loss: 3.5171 - val_accuracy: 0.5909 - val_loss: 2.0884
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5554 - loss: 2.1396 - val_accuracy: 0.5584 - val_loss: 1.5809
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5456 - loss: 1.3937 - val_accuracy: 0.6558 - val_loss: 1.1871
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6124 - loss: 1.1518 - val_accuracy: 0.6299 - val_loss: 1.0718
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6026 - loss: 0.9673 - val_accuracy: 0.6104 - val_loss: 0.9336
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6059 - loss: 0.9087 - val_accuracy: 0.6429 - val_loss: 0.8487
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6091 - loss: 0.8426 - val_accuracy: 0.6753 - 

In [5]:
!pip install keras_tuner


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 5.1 MB/s eta 0:00:00


In [6]:
import keras_tuner as kt

In [24]:
def build_model(hp):
  model = Sequential()
  model.add(Dense(32, activation='relu', input_dim = 8))
  model.add(Dense(1, activation='sigmoid'))

  model.compile(optimizer = hp.Choice('optimizer',values =  ['adam', 'sgd', 'rmsprop', 'adadelta']), loss = 'binary_crossentropy', metrics = ['accuracy'])
  return model

In [25]:
tuner = kt.RandomSearch(build_model, objective = 'val_accuracy', max_trials = 5)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [29]:
tuner.search(X_train, y_train, epochs = 5, validation_data = (X_test, y_test))

In [35]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'rmsprop'}

In [34]:
model = tuner.get_best_models(num_models = 1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [36]:
model.fit(X_train, y_train, batch_size = 32, epochs = 100, initial_epoch = 5, validation_data = (X_test, y_test))

Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 64ms/step - accuracy: 0.6401 - loss: 1.0664 - val_accuracy: 0.6948 - val_loss: 0.9457
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6629 - loss: 0.8962 - val_accuracy: 0.5779 - val_loss: 0.9515
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6661 - loss: 0.8604 - val_accuracy: 0.6688 - val_loss: 1.0943
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6498 - loss: 0.8319 - val_accuracy: 0.6558 - val_loss: 0.9819
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6547 - loss: 0.8619 - val_accuracy: 0.7273 - val_loss: 0.7583
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6661 - loss: 0.7726 - val_accuracy: 0.7273 - val_loss: 0.6985
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6596 - loss: 0.7540 - val_accuracy: 0.7013 - val_loss: 0.7196
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6531 - loss: 0.8016 - val_accuracy: 0.727

GETTING THE BEST U<BER OF UNITS IN THE FIRST LAYER

In [39]:
def build_model(hp):
  model = Sequential()
  units = hp.Int('units', 8, 128, step = 8)
  model.add(Dense(units = units, activation = 'relu', input_dim = 8))
  model.add(Dense(1, activation='sigmoid'))

  model.compile(optimizer = 'rmsprop', loss =  'binary_crossentropy', metrics = ['accuracy'])
  return model

In [42]:
tuner = kt.RandomSearch(build_model, objective = 'val_accuracy', max_trials = 5, directory = 'mydir')

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [43]:
tuner.search(X_train, y_train, epochs = 5, validation_data = (X_test, y_test))

Trial 5 Complete [00h 00m 03s]
val_accuracy: 0.6363636255264282

Best val_accuracy So Far: 0.701298713684082
Total elapsed time: 00h 00m 18s


In [44]:
tuner.get_best_hyperparameters()[0].values

{'units': 40}

In [45]:
model = tuner.get_best_models(num_models = 1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [46]:
model.fit(X_train,y_train, batch_size=32, epochs = 100, initial_epoch = 5, validation_data = (X_test, y_test))

Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 48ms/step - accuracy: 0.6091 - loss: 0.9490 - val_accuracy: 0.6494 - val_loss: 0.9245
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6303 - loss: 0.7974 - val_accuracy: 0.5390 - val_loss: 1.1890
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6156 - loss: 0.8424 - val_accuracy: 0.6688 - val_loss: 0.8797
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6515 - loss: 0.7762 - val_accuracy: 0.6883 - val_loss: 0.7881
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6417 - loss: 0.7583 - val_accuracy: 0.6688 - val_loss: 1.0467
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6498 - loss: 0.7465 - val_accuracy: 0.6558 - val_loss: 0.9144
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6287 - loss: 0.7709 - val_accuracy: 0.5779 - val_loss: 0.9504
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6482 - loss: 0.7121 - val_accuracy: 0.532

GETTING THE BEST NUKBER OF LAYERS

In [48]:
def build_model(hp):
  model  =Sequential()
  model.add(Dense(40, activation= 'relu', input_dim = 8));
  for i in range(hp.Int('num_layers', min_value = 1, max_value = 10)):
    model.add(Dense(40, activation = 'relu'))
  model.add(Dense(1, activation = 'sigmoid'))

  model.compile(optimizer = 'rmsprop', loss = 'binary_crossentropy', metrics = ['accuracy'])
  return model


In [54]:
tuner = kt.RandomSearch(build_model, objective = 'val_accuracy', max_trials = 5, directory = 'mydir2')

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [55]:
tuner.search(X_train, y_train, epochs = 5, validation_data = (X_test, y_test))

Trial 5 Complete [00h 00m 04s]
val_accuracy: 0.6883116960525513

Best val_accuracy So Far: 0.7207792401313782
Total elapsed time: 00h 00m 25s


In [56]:
tuner.get_best_hyperparameters()[0].values

# since the best number of layers is 1 we have already created this model, so no need to create it again

{'num_layers': 1}

FINDING BOTH THE NUMBER OF LAYERS AND THE NUMBER OF NEURONS IN EACH LAYER TOGETHER

In [64]:
def build_model(hp):
    model = Sequential()

    # Tune the number of layers using a loop counter
    for i in range(hp.Int('num_layers', min_value=1, max_value=10)):
        if i == 0:
            model.add(Dense(
                units=hp.Int(f'units_{i}', min_value=8, max_value=128, step=8),
                activation=hp.Choice(f'activation_{i}', values=['relu', 'tanh', 'sigmoid']),
                input_dim=8
            ))
        else:
            model.add(Dense(
                units=hp.Int(f'units_{i}', min_value=8, max_value=128, step=8),
                activation=hp.Choice(f'activation_{i}', values=['relu', 'tanh', 'sigmoid'])
            ))

    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model


    # using f strings because we need to give unique names to each segment in the loop

In [65]:
tuner = kt.RandomSearch(build_model, objective = 'val_accuracy', max_trials = 5, directory = 'mydir3')

In [66]:
tuner.search(X_train, y_train, epochs = 5, validation_data = (X_test, y_test))

Trial 5 Complete [00h 00m 07s]
val_accuracy: 0.6623376607894897

Best val_accuracy So Far: 0.6948052048683167
Total elapsed time: 00h 00m 30s


In [67]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 1,
 'units_0': 128,
 'activation_0': 'sigmoid',
 'units_1': 48,
 'activation_1': 'relu',
 'units_2': 40,
 'activation_2': 'relu',
 'units_3': 16,
 'activation_3': 'tanh',
 'units_4': 8,
 'activation_4': 'relu',
 'units_5': 72,
 'activation_5': 'tanh',
 'units_6': 96,
 'activation_6': 'sigmoid',
 'units_7': 32,
 'activation_7': 'relu',
 'units_8': 112,
 'activation_8': 'sigmoid'}

In [69]:
best_model = tuner.get_best_models(num_models = 1)[0]
best_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         1,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,281 (5.00 KB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 0 (0.00 B)

In [70]:
# only one layer was used in the best model